In [8]:
import yfinance as yf
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go

In [33]:
# Fetch Data 
TICKER = "^GSPC"
PERIOD = "2y"
WINDOW_SIZE = 21

# Focus window
FOCUS_WINDOW = 3 #months

In [34]:
def fetch_sma21(ticker:str="^GSPC", period:str="6mo", window:int=21) -> pd.DataFrame:
    """
    Fetch S&P500 historical data and compute SMA21.
    Returns a DataFrame with 'Close' and 'SMA21'.
    """
    # Extract Data 
    dat = yf.Ticker(ticker)
    historic_data = dat.history(period, interval="1d", rounding=True)

    # Transform Data 
    df = historic_data[["Close"]].copy()
    df[f"sma_{window}"] = df.rolling(window=window).mean()

    df["Diff"] = df["Close"] - df[f"sma_{window}"]
    df["Diff_pct"] = ((df["Close"] - df[f"sma_{window}"]) / df[f"sma_{window}"] * 100).round(1)

    return df

In [35]:
def plot_close_sma_diff(df: pd.DataFrame, window=WINDOW_SIZE, diff_type="Diff_pct"):
    """
    Plot Close and SMA as lines, and Diff or Diff_pct as bars with clean formatting.
    """
    if diff_type not in ["Diff", "Diff_pct"]:
        raise ValueError("diff_type must be 'Diff' or 'Diff_pct'")
    
    # Format x-axis for clean date display
    df_plot = df.copy()
    df_plot.index = df_plot.index.strftime("%Y-%m-%d")

    fig = go.Figure()

    # Lines: Close and SMA
    fig.add_trace(go.Scatter(
        x=df_plot.index, y=df_plot["Close"], mode="lines", name="Close",
        hovertemplate='%{y:.2f} USD<br>%{x}'
    ))
    fig.add_trace(go.Scatter(
        x=df_plot.index, y=df_plot[f"sma_{window}"], mode="lines", name=f"SMA{window}",
        hovertemplate='%{y:.2f} USD<br>%{x}'
    ))

    # Bars: Diff or Diff_pct
    colors = ['red' if val > 0 else 'green' for val in df_plot[diff_type]]
    fig.add_trace(go.Bar(
        x=df_plot.index, y=df_plot[diff_type], name=diff_type, yaxis="y2",
        marker_color=colors, opacity=0.6,
        hovertemplate='%{y:.1f}%<br>%{x}' if diff_type=="Diff_pct" else '%{y:.2f} USD<br>%{x}'
    ))

    # Layout
    fig.update_layout(
        title=f"S&P500 Close vs SMA{window} and {diff_type}",
        xaxis_title="Date",
        yaxis=dict(title="Price (USD)"),
        yaxis2=dict(title=diff_type, overlaying="y", side="right"),
        legend=dict(x=0.01, y=0.99),
        xaxis=dict(tickangle=-45),  # tilt dates for readability
    )

    fig.show()

In [36]:
# Get Data
df = fetch_sma21(period=PERIOD)

# Filter Focus
start_date = df.index.max() - pd.DateOffset(months=FOCUS_WINDOW)
df_focus = df.loc[df.index >= start_date]

# Plot Data
plot_close_sma_diff(df_focus)

In [32]:
# Check the last timestamp
current_diff = df_focus.iloc[-1]
print(current_diff)

Close       6495.150000
sma_21      6446.584286
Diff          48.565714
Diff_pct       0.800000
Name: 2025-09-08 00:00:00-04:00, dtype: float64
